# Lab 2: Monte Carlo Estimation and Importance Sampling

Monte Carlo methods convert intractable integrals into sample averages. This lab implements the estimators from r2 and r5 — expected value, variance, importance sampling, and effective sample size — and builds intuition for when importance sampling helps and when it catastrophically fails.

**Sections**
1. Monte Carlo Estimation and Convergence
2. Linearity of Expectation (Empirical)
3. Importance Sampling
4. Effective Sample Size
5. Weight Collapse Demonstration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
rng = np.random.default_rng(7)
print('Setup complete.')

## 1. Monte Carlo Estimation and Convergence

The MC estimator for $\mathbb{E}_p[f(X)] = \int f(x)\,p(x)\,dx$ is:
$$\hat{\mu}_N = \frac{1}{N}\sum_{i=1}^N f(x_i), \quad x_i \sim p$$

By the Law of Large Numbers $\hat{\mu}_N \to \mathbb{E}[f(X)]$, and error scales as $O(1/\sqrt{N})$.

We estimate $\mathbb{E}[X^2]$ for $X \sim \mathcal{N}(0,1)$. The true value is $\text{Var}(X) + (\mathbb{E}[X])^2 = 1 + 0 = 1$.

In [ ]:
max_N = 100_000
samples = rng.standard_normal(max_N)
f_vals = samples**2  # f(X) = X²

# Running mean
ns = np.unique(np.logspace(1, 5, 200).astype(int))
running_means = [f_vals[:n].mean() for n in ns]
running_errors = [abs(m - 1.0) for m in running_means]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.semilogx(ns, running_means, 'steelblue', lw=1.5)
ax1.axhline(1.0, color='coral', linestyle='--', lw=2, label='true E[X²]=1')
ax1.set_xlabel('N (samples)')
ax1.set_ylabel('Estimate')
ax1.set_title('MC estimate of E[X²]')
ax1.legend()

ax2.loglog(ns, running_errors, 'steelblue', lw=1.5, label='actual error')
ax2.loglog(ns, 1/np.sqrt(ns), 'k--', lw=1, alpha=0.6, label='O(1/√N) reference')
ax2.set_xlabel('N')
ax2.set_ylabel('|error|')
ax2.set_title('Convergence rate')
ax2.legend()

plt.tight_layout()
plt.show()

print(f"N=100: estimate = {f_vals[:100].mean():.4f}")
print(f"N=10000: estimate = {f_vals[:10000].mean():.4f}")
print(f"N=100000: estimate = {f_vals.mean():.6f}  (true: 1.0)")

## 2. Linearity of Expectation (Empirical)

Linearity of expectation holds for **any** random variables — even dependent ones:
$$\mathbb{E}[X + Y] = \mathbb{E}[X] + \mathbb{E}[Y]$$

We test this for $X \sim \text{Uniform}(-1,1)$ and $Y = X^2$ (completely dependent on $X$).

In [ ]:
N = 500_000
X = rng.uniform(-1, 1, N)
Y = X**2

# Analytic values: E[X]=0, E[Y]=E[X²]=1/3, E[X+Y]=1/3
print("Linearity of expectation: E[X+Y] = E[X] + E[Y]")
print(f"  E[X]     = {X.mean():.6f}   (analytic: 0)")
print(f"  E[Y]     = {Y.mean():.6f}   (analytic: 1/3 = {1/3:.6f})")
print(f"  E[X+Y]   = {(X+Y).mean():.6f}   (analytic: 1/3)")
print(f"  E[X]+E[Y]= {X.mean() + Y.mean():.6f}")
print()
print(f"  Match: {abs((X+Y).mean() - (X.mean() + Y.mean())) < 0.001}")
print()
print("Note: X and Y are maximally dependent (Y = X²), yet linearity still holds!")
print(f"  Cov(X,Y) = {np.cov(X, Y)[0,1]:.6f} ≈ 0  (symmetric distribution)")
print(f"  Corr(X,Y)= {np.corrcoef(X, Y)[0,1]:.6f} — uncorrelated but NOT independent")

## 3. Importance Sampling

The IS identity: $\mathbb{E}_p[f(x)] = \mathbb{E}_q\!\left[f(x)\,\frac{p(x)}{q(x)}\right]$.

We estimate $\mathbb{E}_p[x^2]$ for $p = \mathcal{N}(3,1)$ using proposal $q = \mathcal{N}(0,1)$.
True value: $\text{Var}_p(X) + \mu_p^2 = 1 + 9 = 10$.

In [ ]:
p = stats.norm(3, 1)  # target
q = stats.norm(0, 1)  # proposal
f_fn = lambda x: x**2
true_val = 10.0       # E_p[X²] = Var + mean² = 1 + 9

N = 10_000
repeats = 200

direct_estimates = []
is_estimates = []

for _ in range(repeats):
    # Direct sampling from p
    x_p = p.rvs(N, random_state=rng)
    direct_estimates.append(f_fn(x_p).mean())

    # Importance sampling from q
    x_q = q.rvs(N, random_state=rng)
    w = p.pdf(x_q) / q.pdf(x_q)            # importance weights
    is_estimates.append((f_fn(x_q) * w).mean())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, ests, label, color in [
    (axes[0], direct_estimates, 'Direct sampling from p', 'steelblue'),
    (axes[1], is_estimates,    'Importance sampling (q=N(0,1))', 'coral'),
]:
    ax.hist(ests, bins=30, color=color, alpha=0.7, density=True)
    ax.axvline(true_val, color='black', linestyle='--', lw=2, label=f'true={true_val}')
    ax.axvline(np.mean(ests), color='red', linestyle=':', lw=1.5,
               label=f'mean={np.mean(ests):.2f}')
    ax.set_title(label)
    ax.set_xlabel('Estimate of E_p[X²]')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"Direct sampling — mean: {np.mean(direct_estimates):.4f}, std: {np.std(direct_estimates):.4f}")
print(f"Importance sampling — mean: {np.mean(is_estimates):.4f}, std: {np.std(is_estimates):.4f}")
print()
print("IS is unbiased but higher variance — the proposal q barely covers the target p's mass.")

## 4. Effective Sample Size

$$\text{ESS} = \frac{\left(\sum_i w_i\right)^2}{\sum_i w_i^2} \leq N$$

ESS = N when all weights are equal ($p = q$). ESS → 1 when one weight dominates.

In [ ]:
def ess(weights):
    return weights.sum()**2 / (weights**2).sum()

N = 1000
# Vary the mean shift between p and q (same unit variance)
shifts = np.linspace(0, 6, 40)
ess_vals = []

for shift in shifts:
    p_ = stats.norm(shift, 1)
    q_ = stats.norm(0, 1)
    x_q = q_.rvs(N, random_state=rng)
    w = p_.pdf(x_q) / q_.pdf(x_q)
    ess_vals.append(ess(w))

plt.figure(figsize=(9, 4))
plt.plot(shifts, ess_vals, 'steelblue', lw=2)
plt.axhline(N, color='gray', linestyle=':', label=f'N={N} (maximum ESS)')
plt.axhline(N * 0.1, color='coral', linestyle='--', alpha=0.7, label='10% of N')
plt.xlabel('Mean shift |μ_p - μ_q| (in standard deviations)')
plt.ylabel('ESS')
plt.title('Effective Sample Size vs distributional overlap')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Shift=0 (p=q): ESS = {ess_vals[0]:.0f} / {N}  ({ess_vals[0]/N*100:.0f}%)")
print(f"Shift=3σ:      ESS = {ess_vals[20]:.0f} / {N}  ({ess_vals[20]/N*100:.1f}%)")
print(f"Shift=6σ:      ESS = {ess_vals[-1]:.1f} / {N}  ({ess_vals[-1]/N*100:.2f}%)")

## 5. Weight Collapse Demonstration

When the proposal has **lighter tails** than the target, a tiny fraction of samples get enormous weights. ESS collapses to nearly 1, making the estimator unreliable regardless of sample size.

This is why PPO clips importance weights: $\min(r_t, \text{clip}(r_t, 1-\epsilon, 1+\epsilon))$.

In [ ]:
# Target p = N(0, 3) (wide); Proposal q = N(0, 1) (narrow)
# q has lighter tails — some p-regions are rarely sampled → huge weights
p_wide = stats.norm(0, 3)
q_narrow = stats.norm(0, 1)

N = 2000
x_q = q_narrow.rvs(N, random_state=rng)
w = p_wide.pdf(x_q) / q_narrow.pdf(x_q)
w_norm = w / w.sum()  # normalized weights

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Weight distribution — mostly tiny, a few huge
ax1.semilogy(np.sort(w_norm)[::-1], 'steelblue', lw=1.5)
ax1.axhline(1/N, color='gray', linestyle='--', label=f'uniform weight = 1/N')
ax1.set_xlabel('Sample rank (largest first)')
ax1.set_ylabel('Normalized weight (log scale)')
ax1.set_title('Weight distribution — dominated by a few samples')
ax1.legend(fontsize=9)

# Top 10 weights account for most of the total
sorted_w = np.sort(w_norm)[::-1]
cumulative = np.cumsum(sorted_w)
ax2.plot(range(1, N+1), cumulative, 'steelblue', lw=2)
ax2.axhline(0.9, color='coral', linestyle='--', label='90% of total weight')
ax2.set_xlim(0, 100)
ax2.set_xlabel('Number of top samples')
ax2.set_ylabel('Cumulative weight')
ax2.set_title('Cumulative weight from top-k samples')
ax2.legend()

plt.tight_layout()
plt.show()

ess_collapsed = ess(w)
top10_weight = sorted_w[:10].sum()
print(f"ESS = {ess_collapsed:.1f} / {N}  ({ess_collapsed/N*100:.2f}% efficiency)")
print(f"Top 10 samples carry {top10_weight*100:.1f}% of total weight")
print()
print("Even with N=2000, the effective estimator is based on just a handful of points.")
print("This is why PPO clips importance weights — to prevent policy updates from being")
print("dominated by a tiny set of transitions with extreme weights.")